# Customer Segmentation Based on RFM Scores
**This notebook generates custom hardcoded rules for customer segmentation based on RFM scores using Recency, Frequency and Monetary**

### Import requried libraries, config to import the data

In [13]:
import os
import sys
import pandas as pd
import re

sys.path.append(os.path.abspath('..'))

from config import RFM_DATA_PATH

In [7]:
df=pd.read_csv(RFM_DATA_PATH)
df.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40


### Devide the data into 5 groups based on ranks and join them to form RFM scores

In [8]:
df["R"] = pd.qcut(df["Recency"],q=5,labels=[5,4,3,2,1])

df["F"] = pd.qcut(df["Frequency"].rank(method="first"),q=5,labels=[1,2,3,4,5])

df["M"] = pd.qcut(df["Monetary"],q=5,labels=[1,2,3,4,5])


In [9]:
df["rfm_score"] = df[["R","F","M"]].astype(str).agg("".join,axis=1)

In [10]:
df.head()

,CustomerID,Recency,Frequency,Monetary,R,F,M,rfm_score
0,12346.0,326,1,77183.60,1,1,5,115
1,12347.0,2,7,4310.00,5,5,5,555
2,12348.0,75,4,1797.24,2,4,4,244
3,12349.0,19,1,1757.55,4,1,4,414
4,12350.0,310,1,334.40,1,1,2,112


### Define Custome Regex patterns to segment customers into meaningful groups

In [12]:
segmentation = {
    r"^(555|554|545)$":"VIP",
    r"^[45][45].$":"Loyal Customers",
    r"^[45][23].$":"Potential Customers",
    r"51.$":"New Customer",
    r"^[12][45].$":"At Risk",
    r"^(111|121|112)$":"Hibernating"
}

### Customers who do not fall in any of the defined groups are categorized as "Standard Customers"

In [16]:
def customer_segment(dat):
    for patrn,label in segmentation.items():
        if re.fullmatch(patrn,dat):
            return label
    return "Standard Customer"

df["customer_segmentation"] = df["rfm_score"].apply(customer_segment)

In [17]:
df["customer_segmentation"].value_counts()

customer_segmentation
Standard Customer      1926
Loyal Customers         670
Potential Customers     492
VIP                     469
Hibernating             464
At Risk                 275
New Customer             42
Name: count, dtype: int64